In [10]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from huggingface_hub import hf_hub_download
import joblib, torch
import json
from tqdm import tqdm

In [7]:
model_id = "darekpe79/true-false-pbl-herbert"
model = AutoModelForSequenceClassification.from_pretrained(model_id, use_safetensors=True)
tokenizer = AutoTokenizer.from_pretrained(model_id)

enc_path  = hf_hub_download(repo_id=model_id, filename="label_encoder.joblib")
label_enc = joblib.load(enc_path)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 14437.02it/s]
c:\Users\nikod\Documents\main_venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.5.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [21]:
def predict_pbl(input_txt):
    inputs = tokenizer(input_txt, return_tensors="pt", truncation=True, padding=True)
    model.eval()
    with torch.no_grad():
        pred_id = model(**inputs).logits.argmax(-1).item()
    return eval(label_enc.inverse_transform([pred_id])[0])

In [12]:
with open('blogbup_2025-01-30.json', 'r', encoding='utf-8') as jfile:
    page_texts = json.load(jfile)
page_texts = [(e['Link'], e['Tekst artykułu']) for e in page_texts]

In [22]:
results = []
for e in tqdm(page_texts):
    if not e[1]:
        results.append((e[0], False))
    else:
        results.append((e[0], predict_pbl(e[1])))

100%|██████████| 131/131 [03:05<00:00,  1.42s/it]


In [27]:
full_results = len(results)
true_results = len([e for e in results if e[1]])

print(f'''
No. of records: {full_results}
No. of true predictions: {true_results}
% of true predictions: {true_results / full_results * 100:0.2f}
''')


No. of records: 131
No. of true predictions: 60
% of true predictions: 45.80

